# kwargs-pass-through-recipe — ex3: replay back fn via **recipe.kwargs — recover dim at reverse time

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kwargs-pass-through-recipe`. Running the final beacon cell reports progress against the `Backprop: Kwargs pass-through` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Kwargs pass-through` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kwargs-pass-through-recipe`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kwargs-pass-through-recipe"
DD_SUBTOPIC = "Backprop: Kwargs pass-through"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `**recipe.kwargs` — the back fn replays the forward's keyword args

Ex1 threaded kwargs INTO the Recipe; ex2 verified the empty-kwargs edge case. The deepening move is what the Recipe is FOR: at reverse time, you call the back fn with `**recipe.kwargs` so the gradient is computed with the SAME `dim`, `keepdim`, `negative_slope`, etc. the forward used.

```python
# Forward: y = sum(x, dim=1, keepdim=True)
# Stored: recipe.kwargs == {'dim': 1, 'keepdim': True}
# Reverse: grad_in = sum_back(grad_out, y_arr, x_arr, **recipe.kwargs)
#         → sum_back receives dim=1, keepdim=True automatically.
```

**Why this seals the kwargs contract.** Without `**recipe.kwargs` splatting, the back fn would need to inspect `recipe.kwargs['dim']` by hand — error-prone and op-specific. Splatting makes the back fn signature exactly mirror the forward fn signature (minus `grad_out`, `out`, `x`).

**`sum_back` is the canonical example.** `sum` reduces along `dim`. Its gradient must `unsqueeze` along the same `dim` and `expand` back to the input shape. Without the dim from kwargs, `sum_back` would guess — and guess wrong on multi-axis tensors.

### Exercise 3 — replay back fn via **recipe.kwargs — recover dim at reverse time

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `**recipe.kwargs` splat pattern to invoke a back fn (sum_back) using only the Recipe — recover the `dim` and `keepdim` the forward used without storing them anywhere else.
> Keywords: recipe, kwargs, replay, sum-back, dim
> ```

**KCs targeted:** `kwargs-pass-through-recipe`, `back-fn-replays-via-splat`

We give you a `Recipe` dataclass (4 fields: `func`, `args`, `kwargs`, `parents`) and a `sum_back` implementation. Your job is to implement `ex3_replay_sum_back(out_tensor, grad_out)`.

Inputs:
- `out_tensor`: the OUTPUT of a forward `sum`. It carries a `.array` field (the raw torch tensor) and a `.recipe` field (the Recipe stored at forward time). The Recipe's `kwargs` will be something like `{'dim': 1, 'keepdim': True}` or `{}`.
- `grad_out`: a raw torch tensor with the same shape as `out_tensor.array`.

Steps:

1. Recover the parent's raw tensor: `parent_arr = out_tensor.recipe.parents[0].array`.
2. Call `sum_back(grad_out, out_tensor.array, parent_arr, **out_tensor.recipe.kwargs)`.
3. Return the resulting `grad_in` tensor.

The CRITICAL constraint: you must use `**recipe.kwargs` to splat the stored kwargs into the back-fn call — NOT manually unpack or hard-code `dim` / `keepdim`. The whole point is the recipe-replay contract.

In [ ]:
def ex3_replay_sum_back(out_tensor, grad_out):
    """Invoke sum_back using **recipe.kwargs splat to recover dim/keepdim."""
    raise NotImplementedError()


def _test_ex3():
    from dataclasses import dataclass
    from typing import Callable

    @dataclass
    class Recipe:
        func: Callable
        args: tuple
        kwargs: dict
        parents: dict

    class MiniTensor:
        def __init__(self, array, recipe=None):
            self.array = array
            self.recipe = recipe

    def sum_back(grad_out, out, x, **kwargs):
        # Reverse: re-expand grad_out along the reduced dim(s) back to x.shape.
        dim = kwargs.get('dim', None)
        keepdim = kwargs.get('keepdim', False)
        if dim is None:
            return grad_out * t.ones_like(x)
        if not keepdim:
            grad_out = grad_out.unsqueeze(dim)
        return grad_out.expand_as(x)

    # Expose sum_back to the function via __globals__ injection (test scope).
    # In real ARENA code, sum_back is module-level — we mimic that here.
    ex3_replay_sum_back.__globals__['sum_back'] = sum_back

    # === Case A: dim=1, keepdim=False ===
    x = t.arange(12.0).reshape(3, 4)
    kwargs = {'dim': 1, 'keepdim': False}
    out_arr = x.sum(**kwargs)  # shape (3,)
    parent = MiniTensor(x)
    out_tensor = MiniTensor(out_arr, Recipe(t.sum, (x,), kwargs, {0: parent}))
    grad_out = t.ones_like(out_arr)
    grad_in = ex3_replay_sum_back(out_tensor, grad_out)
    expected = t.ones_like(x)  # d(sum)/dx_ij = 1 for each element in the reduced row
    assert grad_in.shape == x.shape, f'shape mismatch: {grad_in.shape}'
    assert t.allclose(grad_in, expected), f'dim=1: {grad_in}'

    # === Case B: dim=0, keepdim=True ===
    x = t.arange(20.0).reshape(4, 5)
    kwargs = {'dim': 0, 'keepdim': True}
    out_arr = x.sum(**kwargs)  # shape (1, 5)
    parent = MiniTensor(x)
    out_tensor = MiniTensor(out_arr, Recipe(t.sum, (x,), kwargs, {0: parent}))
    grad_out = t.ones_like(out_arr)
    grad_in = ex3_replay_sum_back(out_tensor, grad_out)
    expected = t.ones_like(x)
    assert grad_in.shape == x.shape
    assert t.allclose(grad_in, expected)

    # === Case C: empty kwargs (reduce-all) ===
    x = t.arange(6.0).reshape(2, 3)
    kwargs = {}
    out_arr = x.sum()  # scalar
    parent = MiniTensor(x)
    out_tensor = MiniTensor(out_arr, Recipe(t.sum, (x,), kwargs, {0: parent}))
    grad_out = t.tensor(2.0)
    grad_in = ex3_replay_sum_back(out_tensor, grad_out)
    assert grad_in.shape == x.shape
    expected = 2.0 * t.ones_like(x)
    assert t.allclose(grad_in, expected), f'reduce-all: {grad_in}'

    # === Case D: non-unit grad_out scales linearly ===
    x = t.arange(12.0).reshape(3, 4)
    kwargs = {'dim': 1, 'keepdim': False}
    out_arr = x.sum(**kwargs)
    parent = MiniTensor(x)
    out_tensor = MiniTensor(out_arr, Recipe(t.sum, (x,), kwargs, {0: parent}))
    grad_out = t.tensor([1.0, 2.0, 3.0])
    grad_in = ex3_replay_sum_back(out_tensor, grad_out)
    expected = t.tensor([[1.]*4, [2.]*4, [3.]*4])
    assert t.allclose(grad_in, expected), f'scaled grad_out: {grad_in}'

    # === Case E: cross-check vs torch.autograd ===
    x = t.randn(3, 4, requires_grad=True)
    y = x.sum(dim=1, keepdim=False)
    y.sum().backward()
    parent = MiniTensor(x.detach())
    out_tensor = MiniTensor(y.detach(),
                           Recipe(t.sum, (x.detach(),), {'dim': 1, 'keepdim': False}, {0: parent}))
    grad_in = ex3_replay_sum_back(out_tensor, t.ones_like(y))
    assert t.allclose(x.grad, grad_in, atol=1e-6), f'autograd mismatch'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_replay_sum_back(out_tensor, grad_out):
    recipe = out_tensor.recipe
    parent_arr = recipe.parents[0].array
    # Splat the stored kwargs into the back fn — dim, keepdim, etc.
    return sum_back(grad_out, out_tensor.array, parent_arr, **recipe.kwargs)
```

**`**recipe.kwargs` IS the contract.** Without the splat you'd either hard-code keys (`recipe.kwargs.get('dim')`) — coupling the dispatcher to every op's signature — or write per-op dispatchers. Splatting makes one dispatcher work for `sum_back`, `mean_back`, `softmax_back`, etc., as long as each back fn's signature mirrors its forward's.

**Empty kwargs `**{}` is still valid splat syntax.** Python accepts `f(**{})` as 'call with no extra kwargs'. So ex2's empty-kwargs invariant is what makes this drill's replay work uniformly across reduce-all and reduce-along-dim cases.

**`recipe.parents[0]` for unary ops.** Multi-input ops iterate `recipe.parents.items()` and call the appropriate argnum-keyed back fn for each — that's the next drill (parents-dispatch). Here we focus on the kwargs-replay piece in isolation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()